# Single‑Cell RNA Sequencing of Pancreatic Cancer Specimens Using Seurat (FAIR^2 Dataset) Exploration with `mlcroissant`
This notebook provides a practical template for loading and exploring a complex biomedical dataset using the `mlcroissant` library.

### Dataset Source
This dataset is described by a Croissant schema, available via the following URL:

`https://sen.science/doi/10.71728/senscience.g45k-sg4y/fair2.json`

*Tabulated enrichment analysis of Gene Ontology biological process terms for cluster 5 from single-cell RNA sequencing of pancreatic cancer specimens. Dataset includes GO term IDs, term descriptions, gene ratios, enrichment statistics, p-values, adjusted values, gene lists, and counts.*

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.g45k-sg4y/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print out the dataset name and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Review available record sets, fields, and their `@id` values.

All references use the `@id` (unique identifier) for each entity in the dataset. This ensures consistent linkage between fields, columns, and record sets.

Let's enumerate and print all available record sets and their fields (by `@id`).

In [ ]:
# Display available record sets with their @id and fields
record_sets = dataset.metadata.record_sets
record_set_ids = []
for rs in record_sets:
    print(f"RecordSet @id: {rs.id}")
    record_set_ids.append(rs.id)
    print("  Fields:")
    for field in rs.fields:
        print(f"    Field @id: {field.id}, name: {field.name}, data_type: {field.data_type}")
    print("  Columns:")
    for column in rs.columns:
        print(f"    Column @id: {column.id}, name: {column.name}, data_type: {column.data_type}")
    print("---")

## 3. Data Extraction

Load data from the main record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

For this dataset, we pick the main enrichment analysis record set, likely corresponding to biological process GO enrichment for cluster 5. (The exact `@id` is discovered from the previous cell's output.)

In [ ]:
# Select the main record set
main_record_set_id = record_set_ids[0]  # Choose the first one for this example. Adjust if needed.
print(f"Using RecordSet: {main_record_set_id}")

# Load the records from the selected record set
records = list(dataset.records(record_set=main_record_set_id))

# Create a DataFrame
df = pd.DataFrame(records)

# Print column names and show first few rows
print("Fields and Columns (@id) in DataFrame:")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps: filtering records, normalizing numeric fields, and grouping/categorizing data.

Example: Let's filter rows with an enrichment score (e.g. adjusted p-value) less than a threshold, normalize a count column, and group by biological process term.

In [ ]:
# Choose relevant numeric and categorical fields/columns by their @id
# For demonstration, they will be selected from df.columns (as discovered above)

# Example candidates (adjust names with real ones from df.columns)
numeric_field_id = None
group_field_id = None
for col in df.columns:
    if 'p.adjust' in col.lower() or 'adjp' in col.lower() or 'padj' in col.lower():
        numeric_field_id = col  # adjusted p-value
    elif 'gene.count' in col.lower() or 'count' in col.lower():
        count_field_id = col
    elif 'go.term' in col.lower() or 'biological.process' in col.lower():
        group_field_id = col
print(f"Numeric field (adjusted p-value): {numeric_field_id}")
print(f"Count field: {count_field_id}")
print(f"Group field: {group_field_id}")

# Filter for significant enrichments (adjusted p-value < 0.05)
if numeric_field_id:
    threshold = 0.05
    filtered_df = df[df[numeric_field_id] < threshold]
    print(f"Filtered records with {numeric_field_id} < {threshold}:")
    print(filtered_df.head())

    # Normalize the gene count field, if available
    if count_field_id:
        filtered_df[count_field_id + '_normalized'] = (
            filtered_df[count_field_id] - filtered_df[count_field_id].mean()
        ) / filtered_df[count_field_id].std()
        print(f"Normalized {count_field_id} for filtered records:")
        print(filtered_df[[count_field_id, count_field_id + '_normalized']].head())

    # Group by biological process term and show mean adjusted p-value/gene count
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id).agg({numeric_field_id: 'mean', count_field_id: 'mean'})
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())

## 5. Visualization

Visualize distributions such as gene counts per biological process term and adjusted p-values.

These plots help reveal significant enrichment patterns in biological processes for cluster 5.

In [ ]:
# Histogram of gene counts for significant enrichments
if count_field_id and numeric_field_id and group_field_id and not filtered_df.empty:
    plt.figure(figsize=(8, 4))
    plt.hist(filtered_df[count_field_id], bins=15, color='teal', edgecolor='black')
    plt.title('Distribution of Gene Counts for Significant GO Terms')
    plt.xlabel(count_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # Top N biological process terms by gene count
    top_terms = filtered_df.groupby(group_field_id)[count_field_id].mean().sort_values(ascending=False).head(10)
    plt.figure(figsize=(10, 5))
    top_terms.plot(kind='bar', color='coral')
    plt.title('Top 10 GO Biological Process Terms by Mean Gene Count')
    plt.xlabel(group_field_id)
    plt.ylabel('Mean Gene Count')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

    # Scatter plot of gene count vs adjusted p-value
    plt.figure(figsize=(7, 4))
    plt.scatter(filtered_df[count_field_id], filtered_df[numeric_field_id], alpha=0.7, color='purple')
    plt.title('Gene Count vs Adjusted p-value for Significant GO Terms')
    plt.xlabel(count_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()
else:
    print('Visualization fields not found or no filtered records available.')

## 6. Conclusion

In this notebook, we explored the FAIR^2 single-cell RNA-seq enrichment data for cluster 5 from pancreatic cancer specimens. Using `mlcroissant`, we:

- Loaded and examined the Croissant schema metadata.
- Extracted records referencing entities by their unique `@id`.
- Filtered significant GO term enrichments and normalized gene counts.
- Grouped and visualized biological process terms.

This analysis highlighted the most enriched biological processes, supporting downstream interpretation of the tumor microenvironment and cell type assignments.